# Lightweight Alignment of Vision-Language Representations via Embedding Denoising

This notebook implements Hyperparameter ablation studies


## Step: Setup and Dataset Loading


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification
from datasets import load_dataset

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import pandas as pd

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cuda


In [8]:
# Load dataset
dataset = load_dataset("SKyu/my-image-captioning-dataset")

# Consistent dataset splits for all experiments
train_dataset = dataset["train"].select(range(2000, 3000))  # 1000 samples
val_dataset = dataset["train"].select(range(1, 1000))      # 999 samples
test_dataset = dataset["train"].select(range(1000, 2000))  # 1000 samples

print(f"Train: {len(train_dataset)} samples")
print(f"Val: {len(val_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")


Train: 1000 samples
Val: 999 samples
Test: 1000 samples


In [9]:
# Data collation function
def collate_fn(batch):
    images = [item["image"] for item in batch]
    captions = [item["prompt"] for item in batch]
    return images, captions

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


In [10]:
# Load frozen CLIP model (used by both methods)
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = AutoModelForZeroShotImageClassification.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

# Freeze CLIP for embedding extraction
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

# Helper function to get CLIP embeddings
@torch.no_grad()
def get_clip_embeddings(images, captions):
    """Extract normalized CLIP embeddings"""
    inputs = processor(
        text=captions, images=images, return_tensors="pt",
        padding=True, truncation=True, max_length=77
    ).to(device)

    img_embeds = F.normalize(
        clip_model.get_image_features(inputs["pixel_values"]), dim=-1
    )
    text_inputs = {k: v for k, v in inputs.items()
                   if k in ["input_ids", "attention_mask"]}
    txt_embeds = F.normalize(
        clip_model.get_text_features(**text_inputs), dim=-1
    )
    return img_embeds, txt_embeds


## Diffusion Alignment Module - 1D U-Net Architecture


In [11]:
class UNet1DResidualDenoiser(nn.Module):
    """
    1D U-Net Residual Denoiser for CLIP embeddings
    Cross-modal conditioning: uses text to denoise images (and vice versa)
    """
    def __init__(self, embedding_dim=512, hidden_dim=1024):
        super().__init__()

        # Encoder (takes residual + conditioning)
        self.enc1 = nn.Linear(embedding_dim * 2, hidden_dim)  # *2 for cross-modal conditioning
        self.enc2 = nn.Linear(hidden_dim, hidden_dim)

        # Decoder
        self.dec1 = nn.Linear(hidden_dim, hidden_dim)
        self.dec2 = nn.Linear(hidden_dim, embedding_dim)

        self.act = nn.ReLU()

    def forward(self, x_noisy, x_clean=None, cond=None):
        """
        x_noisy: noisy embedding to denoise
        x_clean: clean embedding (for computing residual during training)
        cond: conditional embedding (cross-modal, e.g., text for image denoising)
        """
        # Compute residual noise
        if x_clean is not None:
            residual = x_noisy - x_clean
        else:
            residual = x_noisy

        # Cross-modal conditioning: concatenate residual with condition
        if cond is not None:
            x = torch.cat([residual, cond], dim=-1)
        else:
            x = residual

        # Encoder
        e1 = self.act(self.enc1(x))
        e2 = self.act(self.enc2(e1))

        # Decoder with skip connection
        d1 = self.act(self.dec1(e2) + e1)
        d2 = self.dec2(d1)

        # Residual subtraction: remove predicted noise
        x_denoised = x_noisy - d2
        return x_denoised


In [12]:
# Noise schedule functions
def add_noise_linear(embeddings, t, max_std=0.3):
    """Linear noise schedule: noise_std = t * max_std"""
    noise_std = t * max_std
    return embeddings + torch.randn_like(embeddings) * noise_std

def add_noise_cosine(embeddings, t, max_std=0.3):
    """Cosine noise schedule: more noise at later timesteps"""
    noise_std = max_std * (1 - np.cos(t * np.pi / 2))
    return embeddings + torch.randn_like(embeddings) * noise_std

# Contrastive loss function (for diffusion training)
def contrastive_loss(img_embeds, txt_embeds, temperature=0.07):
    """Contrastive loss using cross-entropy on similarity matrix"""
    img_embeds = F.normalize(img_embeds, dim=-1)
    txt_embeds = F.normalize(txt_embeds, dim=-1)

    logits = img_embeds @ txt_embeds.T / temperature
    labels = torch.arange(img_embeds.size(0), device=img_embeds.device)

    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)

    return (loss_i2t + loss_t2i) / 2


## Step: Evaluation Functions


In [13]:
def evaluate_method(loader, method="raw", model=None, denoiser=None):
    """
    Evaluate a method on the dataset

    Args:
        loader: DataLoader
        method: "raw", "baseline", or "diffusion"
        model: Fine-tuned CLIP model (for baseline)
        denoiser: Trained denoiser (for diffusion)
    """
    recall_at_1, recall_at_3, recall_at_5 = 0, 0, 0
    total = 0
    cosine_sims = []

    for images, captions in loader:
        if method == "raw":
            img_embeds, txt_embeds = get_clip_embeddings(images, captions)

        elif method == "baseline":
            model.eval()
            with torch.no_grad():
                inputs = processor(
                    text=captions, images=images, return_tensors="pt",
                    padding=True, truncation=True, max_length=77
                ).to(device)
                outputs = model(**inputs)
                img_embeds = F.normalize(outputs.image_embeds, dim=-1)
                txt_embeds = F.normalize(outputs.text_embeds, dim=-1)

        elif method == "diffusion":
            denoiser.eval()
            img_embeds, txt_embeds = get_clip_embeddings(images, captions)
            with torch.no_grad():
                denoised_img = F.normalize(
                    denoiser(img_embeds, cond=txt_embeds), dim=-1
                )
                denoised_txt = F.normalize(
                    denoiser(txt_embeds, cond=img_embeds), dim=-1
                )
            img_embeds, txt_embeds = denoised_img, denoised_txt

        # Compute similarity matrix
        sim_matrix = img_embeds @ txt_embeds.T

        # Compute Recall@K
        for i in range(sim_matrix.size(0)):
            scores = sim_matrix[i]
            top1 = scores.topk(1).indices.item()
            top3 = scores.topk(3).indices.tolist()
            top5 = scores.topk(5).indices.tolist()
            true_id = i

            if top1 == true_id:
                recall_at_1 += 1
            if true_id in top3:
                recall_at_3 += 1
            if true_id in top5:
                recall_at_5 += 1
            total += 1

        # Compute cosine similarity
        cosine_sims.append(
            F.cosine_similarity(img_embeds, txt_embeds, dim=-1).mean().item()
        )

    return {
        "Recall@1": recall_at_1 / total * 100,
        "Recall@3": recall_at_3 / total * 100,
        "Recall@5": recall_at_5 / total * 100,
        "CosineMean": np.mean(cosine_sims)
    }


## Step: Hyperparameter Ablation Studies

In [14]:
print("=" * 80)
print("COMBINED HYPERPARAMETER ABLATION")
print("λ₁ / λ₂  | Noise Schedule | Max Noise")
print("=" * 80)

# -----------------------------
# Ablation search space
# -----------------------------
lambda_combinations = [
    (0.5, 0.5), (1.0, 1.0), (1.5, 1.5),
    (1.0, 0.5), (1.0, 1.5),
    (0.5, 1.0), (1.5, 1.0)
]

noise_schedules = ["linear", "cosine"]
max_noise_levels = [0.1, 0.2, 0.3, 0.4]

ablation_results = []

# -----------------------------
# Unified experiment loop
# -----------------------------
for λ1, λ2 in lambda_combinations:
    for schedule in noise_schedules:
        for max_noise in max_noise_levels:

            print(f"\nTesting λ₁={λ1}, λ₂={λ2}, "
                  f"schedule={schedule}, max_noise={max_noise}")

            # Fresh model for fair comparison
            denoiser = UNet1DResidualDenoiser(
                embedding_dim=512,
                hidden_dim=1024
            ).to(device)

            optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-4)

            # -----------------------------
            # Short training (ablation)
            # -----------------------------
            for epoch in range(5):
                denoiser.train()
                for images, captions in train_loader:
                    img_embeds, txt_embeds = get_clip_embeddings(images, captions)
                    t = torch.rand(1).item()

                    if schedule == "linear":
                        noisy_img = add_noise_linear(img_embeds, t, max_noise)
                        noisy_txt = add_noise_linear(txt_embeds, t, max_noise)
                    else:
                        noisy_img = add_noise_cosine(img_embeds, t, max_noise)
                        noisy_txt = add_noise_cosine(txt_embeds, t, max_noise)

                    denoised_img = denoiser(
                        noisy_img, x_clean=img_embeds, cond=txt_embeds
                    )
                    denoised_txt = denoiser(
                        noisy_txt, x_clean=txt_embeds, cond=img_embeds
                    )

                    mse_loss = (
                        λ1 * F.mse_loss(denoised_img, img_embeds) +
                        λ2 * F.mse_loss(denoised_txt, txt_embeds)
                    )

                    cont_loss = contrastive_loss(
                        denoised_img, denoised_txt, temperature=0.07
                    )

                    loss = mse_loss + cont_loss

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

            # -----------------------------
            # Evaluation
            # -----------------------------
            metrics = evaluate_method(
                test_loader,
                method="diffusion",
                denoiser=denoiser
            )

            ablation_results.append({
                "λ₁": λ1,
                "λ₂": λ2,
                "Schedule": schedule,
                "Max Noise": max_noise,
                "Recall@1": metrics["Recall@1"],
                "Recall@5": metrics["Recall@5"],
                "Cosine": metrics["CosineMean"]
            })

            print(f"  Recall@1: {metrics['Recall@1']:.2f}% | "
                  f"Cosine: {metrics['CosineMean']:.4f}")


COMBINED HYPERPARAMETER ABLATION
λ₁ / λ₂  | Noise Schedule | Max Noise

Testing λ₁=0.5, λ₂=0.5, schedule=linear, max_noise=0.1
  Recall@1: 75.50% | Cosine: 0.7204

Testing λ₁=0.5, λ₂=0.5, schedule=linear, max_noise=0.2
  Recall@1: 71.80% | Cosine: 0.6477

Testing λ₁=0.5, λ₂=0.5, schedule=linear, max_noise=0.3
  Recall@1: 81.70% | Cosine: 0.5367

Testing λ₁=0.5, λ₂=0.5, schedule=linear, max_noise=0.4
  Recall@1: 69.00% | Cosine: 0.2543

Testing λ₁=0.5, λ₂=0.5, schedule=cosine, max_noise=0.1
  Recall@1: 86.60% | Cosine: 0.7139

Testing λ₁=0.5, λ₂=0.5, schedule=cosine, max_noise=0.2
  Recall@1: 90.50% | Cosine: 0.6916

Testing λ₁=0.5, λ₂=0.5, schedule=cosine, max_noise=0.3
  Recall@1: 92.00% | Cosine: 0.6353

Testing λ₁=0.5, λ₂=0.5, schedule=cosine, max_noise=0.4
  Recall@1: 90.70% | Cosine: 0.6323

Testing λ₁=1.0, λ₂=1.0, schedule=linear, max_noise=0.1
  Recall@1: 80.50% | Cosine: 0.7085

Testing λ₁=1.0, λ₂=1.0, schedule=linear, max_noise=0.2
  Recall@1: 71.40% | Cosine: 0.6479

Testing 

In [15]:
ablation_df = pd.DataFrame(ablation_results)

print("\n" + "=" * 80)
print("FINAL ABLATION RESULTS")
print("=" * 80)
print(ablation_df.to_string(index=False))

best_idx = ablation_df["Recall@1"].idxmax()
best_cfg = ablation_df.loc[best_idx]

print("\nBEST CONFIGURATION:")
print(best_cfg)



FINAL ABLATION RESULTS
 λ₁  λ₂ Schedule  Max Noise  Recall@1  Recall@5   Cosine
0.5 0.5   linear        0.1      75.5      97.8 0.720422
0.5 0.5   linear        0.2      71.8      97.0 0.647699
0.5 0.5   linear        0.3      81.7      98.7 0.536692
0.5 0.5   linear        0.4      69.0      95.8 0.254327
0.5 0.5   cosine        0.1      86.6      99.3 0.713940
0.5 0.5   cosine        0.2      90.5      99.8 0.691579
0.5 0.5   cosine        0.3      92.0      99.9 0.635278
0.5 0.5   cosine        0.4      90.7      99.7 0.632267
1.0 1.0   linear        0.1      80.5      98.6 0.708542
1.0 1.0   linear        0.2      71.4      96.7 0.647922
1.0 1.0   linear        0.3      82.3      98.6 0.442773
1.0 1.0   linear        0.4      83.6      98.6 0.401826
1.0 1.0   cosine        0.1      86.9      98.7 0.714192
1.0 1.0   cosine        0.2      84.3      99.3 0.711965
1.0 1.0   cosine        0.3      89.4      99.4 0.614843
1.0 1.0   cosine        0.4      87.4      99.1 0.574368
1.5 1.5